<center><img src="./img/pong-thumbnail.png" width="200" alt="Skills Network Logo"  /></center>
  


## Test Contenedores levantados:


## Dockers:

Para asegurarnos de que todos los servicios levantados en los contenedores funcionan correctamente, podemos definir algunos test automáticos y manuales para verificar cada servicio. A continuación están los pasos y comandos que hemos incluido para testear los contenedores y garantizar que todo está funcionando como se espera:

- ## 1. Verificar el estado de los contenedores
Después de ejecutar `make`, puedos usar este comando para verificar que todos los contenedores estén en ejecución y sin errores:

In [ ]:
make ps
docker compose -f ./src/docker-compose.yml ps
docker ps -a

Asegurarse de que todos los contenedores estén en estado "Up". Si hay alguno en estado "Exited" o "Restarting", eso indicaría un problema con ese servicio.

- ## 2. Comprobación de los logs:
Puedes revisar los logs de cada servicio para ver si se están generando errores o advertencias que no deberían estar allí. Esto es útil para depurar rápidamente problemas de configuración o dependencias faltantes:

In [ ]:
make logs
docker compose -f ./src/docker-compose.yml logs

Podemos filtrar los logs por servicio específico:

In [ ]:
make logs_service
Por favor, especifica un servicio. Uso: make logs_service SERVICE=<nombre_del_servicio>
make logs_service SERVICE=<nombre_del_servicio>
docker compose -f ./src/docker-compose.yml logs <service_name>

- ## 3. Tests específicos por servicio
    - ## a. SQLite (Base de datos):

    Una vez que el contenedor esté levantado, podemos ejecutar comandos SQL para verificar si la base de datos funciona correctamente. Por ejemplo, podemos conectarnos al contenedor de SQLite y listar las tablas para asegurarte de que está configurado:
    - **Test:** Verificar que SQLite está en ejecución
    
- **Comandos:** 

In [ ]:
docker exec -it sqlite sh

/var/lib/sqlite # ls
init_db.sh  sqlite.db

sqlite3 sqlite.db
sqlite>

sqlite> .tables
test

Si obtenemos un listado correcto de tablas, la base de datos está funcionando bien.

Esta vez vemos el scrip que crea la base de datos al levnatar el contenedor y una lista llamada 'test' de prueba.

 - ### Test: Verificar persistencia de datos
 
Insertar un dato en la base de datos y reiniciar el contenedor.



In [ ]:
CREATE TABLE test (id INTEGER PRIMARY KEY, name TEXT);
INSERT INTO test (name) VALUES ('test_entry');

Si ya tenemos una tabla llamada test y nuestro SQLite funciona OK, nos debe de lanzar este error:

In [ ]:
Parse error: table test already exists
  CREATE TABLE test (id INTEGER PRIMARY KEY, name TEXT); INSERT INTO test (name)
               ^--- error here

Por lo tanto cambiaremos nuestro comando para que nos cree la tabla si no existe y que inserte un dato:

In [ ]:
CREATE TABLE IF NOT EXISTS test (id INTEGER PRIMARY KEY, name TEXT);
INSERT INTO test (name) VALUES ('test_entry');

Para ver los datos que hemos insertado ejecutamos este comando:

In [ ]:
SELECT * FROM test;

sqlite> SELECT * FROM test;
1|test_entry
sqlite> .exit
/var/lib/sqlite # exit

Tumbamos y volvemos a levantar el contenedor. Realizamos de nuevo los pasos y el dato debe de serguir en la tabla.

Ahora podemos borrar la tabla con:

In [ ]:
DROP TABLE test;

***
***

- ## b. Servicio Backend (Node.js API):

**Test: Verificar que la API está en ejecución**

- Comando:

    ```yaml
    curl http://localhost:3000/
    ```
- Acción: Realizar una petición GET al endpoint base del backend.
- Resultado esperado: La API debería devolver una respuesta válida, como un mensaje de bienvenida o un JSON con información básica.

    ```yaml
    ¡Hola, mundo desde Node.js!% 
    ```

**Test: Conexión con SQLite**  <font color="green">**(ERROR - Solucionado)**</font>

- Comando:

    ```yaml
    curl http://localhost:3000/api/test_db
    ```

- Nos devuelve un ERROR:
    ```yaml
    <!DOCTYPE html>
    <html lang="en">
    <head>
    <meta charset="utf-8">
    <title>Error</title>
    </head>
    <body>
    <pre>Cannot GET /api/test_db</pre>
    </body>
    </html>
    ```
- Acción: Endpoint que realice una consulta simple a SQLite, como obtener una lista de datos.
- Resultado esperado: Deberías recibir un JSON con los datos almacenados en la base de datos SQLite.

**Test: Verificar persistencia de datos del backend**

- Comando: Insertar un dato en el sistema mediante un endpoint POST y luego reiniciar el contenedor.

In [ ]:
curl -X curl -X POST http://localhost:3000/api/create -d '{"username":"test_user", "email":"test@example.com", "password":"123456"}' -H "Content-Type: application/json"

- Verificar con un GET

In [ ]:
curl http://localhost:3000/api/items

- Resultado esperado: El dato debería persistir después de reiniciar.

- Reconstruir el contenedor backend: Después de realizar estos cambios, necesitas reconstruir tu contenedor para aplicar los cambios en el código.

In [ ]:
make logs_service SERVICE=backend

app  | 
app  | > backend@1.0.0 start
app  | > node app.js
app  | 
app  | Servidor escuchando en http://localhost:3000
app  | Base de datos inicializada correctamente

- Probar las rutas:
    - Usar curl para probar las rutas y asegurarnos de que las consultas están funcionando correctamente. Por ejemplo:

- Obtener todos los usuarios:
- Obtener todos los juegos:
- Probar la conexión a la base de datos:

In [ ]:
curl http://localhost:3000/api/users
[]% 

curl http://localhost:3000/api/games
[]%

curl http://localhost:3000/api/test_db

Conexión a la base de datos exitosa%  

# Solución de error en la comunicación Backend - SQLite

## Descripción del problema

Teníamos un error en la comunicación entre el backend (Node.js) y la base de datos SQLite. El backend no podía crear nuevos ítems en la base de datos, lo que se manifestaba con el error "Cannot POST /api/items" al intentar enviar una solicitud POST a la API.

Además, el contenedor `sqlite` mostraba un error en sus logs: `/var/lib/sqlite/init_db.sh: line 8: can't open /var/lib/sqlite/init.sql: no such file`. Sin embargo, este error era engañoso (un "falso positivo"), ya que la base de datos se creaba correctamente gracias al backend.

## Diagnóstico

1. **Verificación de logs:**
   - Los logs del backend mostraban el mensaje "Conectado a la base de datos SQLite", lo que indicaba que la conexión inicial era exitosa.
   - Los logs del contenedor `sqlite` mostraban el error mencionado anteriormente.

2. **Inspección del contenedor `sqlite`:**
   - Ejecutamos `docker exec -it sqlite sh` para acceder al contenedor `sqlite`.
   - Dentro del contenedor, ejecutamos `sqlite3 sqlite.db` y luego `SELECT * FROM items;` para verificar el contenido de la base de datos.
   - Observamos que los datos iniciales (insertados desde `init.sql`) estaban presentes, lo que confirmaba que la base de datos se había creado correctamente.

3. **Análisis del Dockerfile del backend:**
   - Observamos que el Dockerfile del backend copiaba el archivo `init.sql` a la ruta `/var/lib/sqlite` dentro del contenedor backend: `COPY tools/init.sql /var/lib/sqlite/init.sql`.
   - Esto explicaba por qué el backend podía crear la base de datos, pero también por qué el contenedor `sqlite` no encontraba `init.sql` en su propio sistema de archivos.

## Solución

1. **Implementación del endpoint POST en el backend:**
   - Implementamos la lógica necesaria en el backend (Node.js con Express) para manejar las solicitudes POST a `/api/items`.
   - Esto incluyó:
     - Uso de `body-parser` para parsear el cuerpo de las solicitudes JSON.
     - Validación de los datos recibidos (`name` y `description`).
     - Ejecución de la consulta SQL `INSERT` para crear el nuevo ítem.
     - Envío de una respuesta JSON con el nuevo ítem y su ID.

2. **Modificación del script `init_db.sh`:**
   - Modificamos el script `init_db.sh` en el contenedor `sqlite` para que solo intente inicializar la base de datos si el archivo `sqlite.db` no existe:

     ```sh
     if [ ! -f /var/lib/sqlite/sqlite.db ]; then
         # ... (código para crear la base de datos)
     fi
     ```

   - Esto evita el error "falso positivo" en los logs del contenedor `sqlite`.

# Tests PERSISTENCIAS DATOS EN VOLUMENES PERSISTENTES:

1. **Test de errores backend:**
```yaml
docker logs app
```
   - Verificamos los logs del backend para confirmar el mensaje "Conectado a la base de datos SQLite".
```yaml
> backend@1.0.0 start
> node app.js

Servidor escuchando en http://localhost:3000
Conectado a la base de datos SQLite
Base de datos inicializada correctamente
```
2. **Test existencia de base de datos backend en contenedor sqlite:**
```yaml
 docker exec -it sqlite sh
```
   - Verificamos que existe sqlite.db y que además contiene datos".
```yaml      
/var/lib/sqlite # sqlite3 sqlite.db 
SQLite version 3.48.0 2025-01-14 11:05:00
Enter ".help" for usage hints.
sqlite> SELECT * FROM items;
1|Item 1|Descripción del Item 1
2|Item 2|Descripción del Item 2
sqlite> .exit
/var/lib/sqlite # exit

```
3. **Test de logs de sqlite:**
```yaml
docker logs sqlite
```
   - Verirficamos que no hay errores y las tablas existen en el volumen persistente.
```yaml              
La base de datos ya existe. No se requiere inicialización.
total 4
-rwxr-xr-x    1 root     root           442 Feb 23 11:58 init_db.sh
```
4. **Test verificación ubicación correcta:**
```yaml
docker volume inspect sqlite_data
```
```yaml   
[
    {
        "CreatedAt": "2025-02-23T11:59:36Z",
        "Driver": "local",
        "Labels": {
            "com.docker.compose.config-hash": "4f9528cdc7d7a1788d2ceadc8fe3ecfe7960c88afb1cd44b5a62996854c8de77",
            "com.docker.compose.project": "src",
            "com.docker.compose.version": "2.32.4",
            "com.docker.compose.volume": "sqlite_data"
        },
        "Mountpoint": "/var/lib/docker/volumes/sqlite_data/_data",
        "Name": "sqlite_data",
        "Options": {
            "device": "/Users/usuario/goinfre/data/sqlite",
            "o": "bind",
            "type": "none"
        },
        "Scope": "local"
    }
]

```

5. **Test de creación de ítems:**
   - Enviamos una solicitud POST a `/api/items` con datos válidos (`name` y `description`).
   - Verificamos que la respuesta contenga el nuevo ítem con un ID generado.
   - Consultamos la base de datos directamente (usando `docker exec` y `sqlite3`) para confirmar que el nuevo ítem se ha creado.
   ```yaml
    curl -X POST -H "Content-Type: application/json" -d '{"name": "Item 3", "description": "CREADO POR DAVID"}' http://localhost:3000/api/items {"id":8,"name":"Item 3","description":"CREADO POR DAVID"}
   ```
   - Verificamos que se han creado:
   ```yaml
   curl http://localhost:3000/api/items 

   [{"id":1,"name":"Item 1","description":"Descripción del Item 1"},{"id":2,"name":"Item 2","description":"Descripción del Item 2"},{"id":3,"name":"Item 1","description":"Descripción del Item 1"},{"id":4,"name":"Item 2","description":"Descripción del Item 2"},{"id":5,"name":"Item 3","description":"Nueva descripción"},{"id":6,"name":"Item 1","description":"Descripción del Item 1"},{"id":7,"name":"Item 2","description":"Descripción del Item 2"},{"id":8,"name":"Item 3","description":"CREADO POR DAVID"},{"id":9,"name":"Item 1","description":"Descripción del Item 1"},{"id":10,"name":"Item 2","description":"Descripción del Item 2"}]%     
   ```

6. **Test de persistencia:**
   - Reiniciamos los contenedores (`make re`).
   - Consultamos la base de datos (directamente o a través de la API) para verificar que los ítems persisten después del reinicio.
   ```yaml
   curl http://localhost:3000/api/items
   ```

4. **Test de `init_db.sh`:**
   - Verificamos los logs del contenedor `sqlite` para confirmar que ya no aparece el error relacionado con `init.sql`.
   - Esto asegura que el script `init_db.sh` está funcionando correctamente y evitando la reinicialización de la base de datos.


***
***

- ## c. Servicio PHP:

**Test: Verificar que PHP está en ejecución** <font color="green">**(ERROR - Solucionado)**</font>

 - Comando:
    ```yaml
    curl http://localhost:8080/
    ```
- Nos devuelve un error:
    ```yaml
    curl: (56) Recv failure: Connection reset by peer
    ```
- Acción: Realizar una petición GET al servidor PHP.
- Resultado esperado: Debería devolver una página PHP o un mensaje de bienvenida si está correctamente configurado.

In [ ]:
curl http://localhost:8080/

**Test: Conexión con Backend** <font color="green">**(SOLUCIONADO)**</font>

- Comando: Crear archivo test en PHP que haga una petición al backend (Node.js).
    ```yaml
    <?php
    $response = file_get_contents('http://backend:3000/api/test');
    echo $response;
    ?>
    ```
-  Acción: Acceder a esta página desde el navegador.
- Resultado esperado: El servidor PHP debería devolver el resultado de la API del backend.


In [ ]:
curl http://localhost:3000/api/test

**Test: logs**

- Comando:
    ```yaml
    make logs_service SERVICE=php
    ```
-  Acción: Acceder a los registros de del servidor.
- Resultado esperado: El servidor PHP debería devolver los logs.


In [ ]:
php  | AH00558: apache2: Could not reliably determine the server's fully qualified domain name, using 172.18.0.7. Set the 'ServerName' directive globally to suppress this message
php  | AH00558: apache2: Could not reliably determine the server's fully qualified domain name, using 172.18.0.7. Set the 'ServerName' directive globally to suppress this message
php  | [Tue Feb 11 17:04:47.086036 2025] [mpm_prefork:notice] [pid 1:tid 1] AH00163: Apache/2.4.62 (Debian) PHP/8.1.31 configured -- resuming normal operations
php  | [Tue Feb 11 17:04:47.086178 2025] [core:notice] [pid 1:tid 1] AH00094: Command line: 'apache2 -D FOREGROUND'
php  | 192.168.65.1 - - [11/Feb/2025:17:06:14 +0000] "GET / HTTP/1.1" 200 74853 "-" "curl/8.7.1"
php  | 192.168.65.1 - - [11/Feb/2025:17:06:41 +0000] "GET / HTTP/1.1" 200 23033 "-" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36"
php  | 192.168.65.1 - - [11/Feb/2025:17:06:42 +0000] "GET /favicon.ico HTTP/1.1" 404 489 "http://localhost:8080/" "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/133.0.0.0 Safari/537.36"
php  | 192.168.65.1 - - [11/Feb/2025:17:11:34 +0000] "GET / HTTP/1.1" 200 74853 "-" "curl/8.7.1"

El mensaje:

In [ ]:
AH00558: apache2: Could not reliably determine the server's fully qualified domain name, using 172.18.0.7. Set the 'ServerName' directive globally to suppress this message

Es una advertencia común en Apache.

No afecta el funcionamiento del servidor, pero podemos eliminarla configurando la directiva ServerName en la configuración de Apache.

Solución: <font color="red">**(POR SOLUCIONAR)**</font>

- ### Test de Seguridad - Servicio PHP: (<font color="red">**(usuario: www-data)**</font>)

**Configuración Dockerfile** <font color="green">**(ERROR - Solucionado)**</font>

 - Dockerfile:

In [ ]:
FROM php:8.1-apache

RUN docker-php-ext-install pdo pdo_mysql mysqli

# Crear el archivo test.php
RUN echo '<?php \
$response = file_get_contents("http://backend:3000/api/test"); \
echo $response; \
?>' > /var/www/html/test.php

COPY . /var/www/html

RUN chown -R www-data:www-data /var/www/html \
    && chmod -R 755 /var/www/html

EXPOSE 80

CMD ["apache2-foreground"]

### Línea problemática:

- Explicación de la línea:
```yaml
RUN chown -R www-data:www-data /var/www/html \ && chmod -R 755 /var/www/html
```

1. **`chown -R www-data:www-data /var/www/html`:**

    - `chown`: Este comando cambia el propietario de archivos o directorios. En este caso, el parámetro -R indica que debe hacerlo de forma recursiva, es decir, aplicarlo también a todos los archivos y subdirectorios dentro de `/var/www/html`.
    - `www-data:www-data`: Esto significa que el usuario propietario y el grupo propietario del directorio `/var/www/html` serán `www-data`, que es el usuario y grupo predeterminado bajo el cual Apache ejecuta sus procesos.
    - Esto garantiza que el servidor web Apache (que se ejecuta como `www-data`) tenga los permisos necesarios para acceder, modificar y servir los archivos del directorio `/var/www/html`.

2. **`chmod -R 755 /var/www/html`**:

    - `chmod`: Cambia los permisos de archivos o directorios.
    - `755`: Esto significa que el propietario (en este caso, `www-data`) tiene permisos de lectura, escritura y ejecución (7), mientras que los demás usuarios y grupos tienen permisos de lectura y ejecución (5).
    - Este comando también es recursivo (`-R`), por lo que todos los archivos y directorios dentro de `/var/www/html` tendrán estos permisos.

### Problemas de Seguridad
**Aunque esta configuración asegura que Apache puede acceder a los archivos, no es la mejor práctica de seguridad por varias razones:**

1.  #### Permisos 755 en todos los archivos:
    - Dar permisos de ejecución (`x`) a archivos y directorios no siempre es necesario o deseable. **En particular, los archivos PHP y otros archivos estáticos no deberían ser ejecutables.**
2. #### Ejecutar con `www-data` sin aislamiento de usuarios:
    - Aunque www-data es un usuario sin privilegios en el contenedor, es mejor crear un usuario específico para tu aplicación, ya que esto mejora el aislamiento y la seguridad.
3. #### Eliminación de archivos como root:
    - Hemos comprobado, que si cambiamos el propietario de los archivos a `www-data` y luego intentamos manipularlos fuera del contenedor, puede que necesitemos permisos de root, lo que es incómodo y puede causar problemas de seguridad. Y en 42 NO tenemos permisos root, por lo que no podemos borrar los volumenes de archivos persistentes.

### Alternativa: Crear un usuario específico
En lugar de usar el usuario `www-data` por defecto, creamos un nuevo usuario en el contenedor y le damos los permisos adecuados para mejorar la seguridad.

### Problema: Volúmenes montados en el host sin permisos root

- Cuando montas volúmenes desde el host al contenedor, los archivos dentro del volumen se conservan con los permisos y propietarios originales del host. Si esos archivos pertenecen a `root` o cualquier otro usuario que no es el tuyo, tendremos problemas al intentar modificarlos o eliminarlos en el host sin privilegios de `root`.

### Solución para evitar el uso de sudo al manipular volúmenes en el host:

In [ ]:
FROM php:8.1-apache

# Instalar dependencias necesarias para compilar extensiones PHP
RUN apt-get update && apt-get install -y \
    libfreetype6-dev \
    libjpeg62-turbo-dev \
    libpng-dev \
    libonig-dev \
    libxml2-dev \
    && docker-php-ext-configure gd --with-freetype --with-jpeg \
    && docker-php-ext-install -j$(nproc) pdo pdo_mysql mysqli gd

RUN docker-php-ext-install pdo pdo_mysql mysqli

# Crear un nuevo usuario en el contenedor con tu mismo UID y GID
ARG UID=1000   # Sustituye esto por tu UID real
ARG GID=1000   # Sustituye esto por tu GID real
RUN groupadd -g $GID myuser && useradd -m -u $UID -g $GID myuser

# Crear el archivo test.php
RUN echo '<?php \
$response = file_get_contents("http://backend:3000/api/test"); \
echo $response; \
?>' > /var/www/html/test.php

COPY . /var/www/html

#RUN chown -R www-data:www-data /var/www/html \
#    && chmod -R 755 /var/www/html

# Cambiar la propiedad de los archivos al nuevo usuario
RUN chown -R myuser:myuser /var/www/html

# Cambiar al nuevo usuario no privilegiado
USER myuser

EXPOSE 80

CMD ["apache2-foreground"]



### Explicación:

1. Usar el mismo UID/GID que tu usuario del host:

    - Cuando creamos el usuario en el contenedor, nos aseguramos de que tiene el mismo UID (User ID) y GID (Group ID) que nuestro usuario en el host. De esta manera, los archivos creados dentro del contenedor aparecerán en el host con el propietario correcto y no necesitarás permisos de root para manejarlos.

In [ ]:
id -u   # Obtiene tu User ID (UID)
id -g   # Obtiene tu Group ID (GID)

Cambiamos los valores UID y GID en ARG UID=1000 y ARG GID=1000 por los valores que obtuviste con id -u e id -g en tu host.

### Resultado esperado:

- **Archivos con permisos correctos:** Como el contenedor y tu host ahora comparten el mismo UID/GID, los archivos creados o modificados dentro del contenedor aparecerán en el host con los permisos correctos, sin necesidad de sudo para modificarlos o eliminarlos.
- **Evitar problemas con root:** Al evitar el uso de root en la manipulación de archivos y volúmenes, ya no deberías enfrentarte a problemas de permisos cuando accedes a esos archivos desde el host.
### Ventajas de este enfoque:
- **Sin necesidad de sudo:** Los archivos serán manipulables sin necesidad de usar sudo, ya que tendrán los permisos de tu usuario desde el principio.
- **Mantenimiento sencillo:** Puedes seguir usando volúmenes sin preocuparte por permisos, ya que tu usuario del host será el propietario de los archivos creados/modificados.

### Ventajas de seguridad significativas:
1. Principio de privilegios mínimos: 
- En cualquier sistema, se debe aplicar el principio de "privilegios mínimos", es decir, que los procesos y usuarios solo deben tener los permisos necesarios para realizar sus tareas, sin acceso adicional innecesario.

- **Ventaja**: Si el contenedor está ejecutándose bajo un usuario no privilegiado, en lugar de root, limita el impacto de posibles vulnerabilidades o fallos en la aplicación. Si un atacante compromete el contenedor, tendría solo los permisos del usuario sin privilegios y no podría ejecutar acciones administrativas o críticas en el sistema anfitrión.

2. Mitigación de ataques de escalada de privilegios
- **Descripción**: Si los archivos en el volumen montado pertenecen a root, o si los procesos en el contenedor están ejecutándose como root, un atacante que logre comprometer el contenedor podría intentar explotar esa situación para obtener acceso de mayor nivel.

- **Ventaja**: Ejecutar el contenedor con un usuario no privilegiado y asignar correctamente permisos a los archivos evita que un atacante pueda escalar sus privilegios. Si algo va mal, los daños quedan limitados a lo que el usuario no privilegiado pueda hacer, reduciendo el riesgo de acceso a archivos críticos o la configuración del sistema.


***
***

- ## d. Servicio Frontend (TypeScript + Tailwind):

**Test: Verificar que el frontend está en ejecución** <font color="green">**(SOLUCIONADO)**</font>

 - Comando:
    ```yaml
    curl http://localhost:3001/
    ```
 - No devuelve nada:
    ```yaml
    
    ```
- Acción: Realizar una petición GET al servidor frontend.
- Resultado esperado: El frontend debería devolver la aplicación web inicial.

In [ ]:
curl http://localhost:3001/

## **Test: Comunicación con Backend** <font color="red">(ERROR)</font>

 - Acción: Asegurarse de que la aplicación frontend puede realizar peticiones al backend.
- Resultado esperado: El frontend debe mostrar los datos que provienen de la API de backend (por ejemplo, una lista de ítems).

# Notebook Jupyter: Tests de comunicación Frontend - Backend <font color="yellow">(ESTOY CON ESTO)</font>

## Objetivo

Verificar la correcta comunicación entre el frontend y el backend de la aplicación. Esto incluye la conexión inicial, la solicitud y envío de datos, la visualización de datos y el manejo de errores.

## Puntos clave a probar

1. **Conexión inicial:**
   - ¿Se carga la aplicación frontend sin errores de conexión en la consola del navegador?
   - ¿La URL del backend (ej: `http://localhost:3000`) es accesible desde el frontend?

2. **Solicitud de datos (GET):**
   - ¿El frontend puede realizar solicitudes GET a la API del backend?
   - ¿La respuesta del backend contiene los datos esperados (ej: lista de ítems en JSON)?
   - ¿Se visualizan los datos correctamente en el frontend?

3. **Envío de datos (POST):**
   - ¿El frontend puede enviar solicitudes POST a la API del backend con datos (ej: para crear un nuevo ítem)?
   - ¿El backend procesa los datos correctamente y los guarda en la base de datos?
   - ¿Se refleja el cambio en la base de datos y en la interfaz del frontend?

4. **Manejo de errores:**
   - ¿El frontend maneja correctamente los errores de conexión o de la API (ej: errores 500, 404)?
   - ¿Se muestran mensajes de error apropiados al usuario?

## Pasos para realizar los tests

1. **Carga inicial del frontend:**
   - Abre la aplicación frontend en tu navegador: Aseguramo de que el frontend esté en ejecución (por ejemplo, en `http://localhost:3001`).
   - Abre la consola de desarrollador - Presiona F12 o Ctrl + Shift + I (Windows/Linux) o Cmd + Option + I (Mac).
   - Verifica que no haya errores de conexión o CORS en la consola. <font color="green">**(SOLUCIONADO)**</font>
   ```yaml
   main-a28abbcc.js:40 Uncaught Error: Minified React error #299; visit https://reactjs.org/docs/error-decoder.html?invariant=299 for the full message or use the non-minified dev environment for full errors and additional helpful warnings.
    at ye.createRoot (main-a28abbcc.js:40:55371)
    at main-a28abbcc.js:40:57976
   ```
   - <font color="red">El error Minified React error #299 ocurre cuando:</font>
      - Estás usando una versión de React 18 o superior.
      - No estás utilizando correctamente `ReactDOM.createRoot` para renderizar tu aplicación.
      - Hay un problema con la configuración del punto de entrada de tu aplicación.
   - Si hay errores, revisa la configuración de conexión del frontend y la configuración CORS del backend.

2. **Solicitud de datos (GET):**
   - Identifica la URL del endpoint de la API que el frontend utiliza para obtener datos (ej: `GET /api/items`).
   - En la consola de desarrollador, ve a la pestaña "Red" o "Network".
   - Recarga la página o realiza la acción en el frontend que debería desencadenar la solicitud GET.
   - Inspecciona la solicitud GET en la lista.
   - Verifica:
     - El código de estado de la respuesta debe ser 200 (OK).
     - La respuesta debe estar en formato JSON.
     - La respuesta debe contener los datos esperados.
   - Si hay errores, revisa la implementación del endpoint en el backend y la lógica de solicitud en el frontend.

3. **Envío de datos (POST):**
   - Identifica la URL del endpoint de la API que el frontend utiliza para enviar datos (ej: `POST /api/items`).
   - En el frontend, realiza la acción que debería enviar los datos (ej: llenar un formulario y enviarlo).
   - En la consola de desarrollador, inspecciona la solicitud POST.
   - Verifica:
     - El código de estado de la respuesta debe ser 200 (OK) o 201 (Created).
     - La respuesta puede contener el nuevo recurso creado.
   - En el backend, verifica que los datos se hayan guardado correctamente en la base de datos.
   - Si hay errores, revisa la implementación del endpoint POST en el backend, la lógica de envío en el frontend y la configuración de la base de datos.

4. **Manejo de errores:**
   - Simula un error en el backend (ej: desconecta la base de datos, cambia la URL de la API a una incorrecta).
   - Realiza una acción en el frontend que debería comunicarse con el backend.
   - Verifica que el frontend muestre un mensaje de error apropiado (ej: "Error de conexión", "Error al obtener los datos").
   - Si el manejo de errores no es correcto, revisa la lógica de manejo de errores en el frontend y asegúrate de que se capturen y manejen correctamente los errores de la API.

## Ejemplo de código (frontend - React)

```javascript
// Ejemplo de solicitud GET con fetch
fetch('/api/items')
  .then(response => {
    if (!response.ok) {
      throw new Error(`HTTP error! status: ${response.status}`);
    }
    return response.json();
  })
  .then(data => {
    // Mostrar los datos en el frontend
    console.log(data);
  })
  .catch(error => {
    // Mostrar un mensaje de error al usuario
    console.error('Error al obtener los datos:', error);
    alert('Error al obtener los datos. Por favor, inténtalo de nuevo más tarde.');
  });

***
***

- ## e. Servicio Blockchain (avalanche):

## **Test: Verificar que avalache está en lavantado** <font color="green">**(SOLUCIONADO)**</font>

 - Comando:
    ```yaml
    make ps
    ```
 - Servicio no está levatado:
    ```yaml
    Restarting (1) 18 seconds ago       
    ```
- Buscamos los posibles errores con el Comando:
   ```yaml
   make logs_service SERVICE=avalanche
   ````
- Devuelve este error:
   ```yaml
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   blockchain  | exec /avalanchego/avalanchego: no such file or directory
   ```
- El error `"exec /avalanchego/avalanchego: no such file or directory"` indica que el binario de AvalancheGo no se encuentra en la ruta especificada (/avalanchego/avalanchego) dentro del contenedor 

In [ ]:
docker exec -it blockchain ls -l /avalanchego/avalanchego

- Listamos que hay dentro del contenedor y nos devuleve este error:
```yaml
Error response from daemon: Container b5ea0a2e15b059f91b3666984e10ffa47cab5e7eb890abd936bb1b4c9ba3c8e5 is restarting, wait until the container is running
 ```

No podemos entrar en el contenedor porque no está lavantado

 - Comando:
    ```yaml
    make ps
    ```
- Respuesta:
    ```yaml
     avalanche   56 seconds ago   Up 37 seconds  0.0.0.0:9650-9651->9650-9651/tcp
    ```
### Acceder al contenedor de `avalanche`:

In [ ]:
docker exec -it blockchain sh  
ls

## Test: Verificar que Avalanche está en ejecución <font color="red">**(ERROR)**</font>
        
- Comando:

In [ ]:
curl http://localhost:9650/ext/health

### Acción:
- Realizar una petición GET al endpoint de salud del nodo de Avalanche.
### Resultado esperado: 
- Un JSON con el estado healthy: true si el nodo está funcionando correctamente.
### <font color="red">Resultado RECIBIDO</font> : 
```yaml
curl: (56) Recv failure: Connection reset by peer
```

## Test: Comunicación con Backendn <font color="red">**(ERROR)**</font>
        
- Comando:

In [ ]:
curl http://localhost:3000/api/avalanche_status

### Acción:
- Crear un endpoint en el backend que interactúe con Avalanche
### Resultado esperado: 
- El backend debería devolver información relevante del estado de la blockchain.
### <font color="red">Resultado RECIBIDO</font> : 
```yaml
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Error</title>
</head>
<body>
<pre>Cannot GET /api/avalanche_status</pre>
</body>
</html>
```

***
***

- ## f. Servicio OWASP ZAP (Seguridad):

## **Test: Verificar que OWASP ZAP está en ejecución** <font color="red">**(ERROR por SOLUCIONAR)**</font>

- Comando:

```yaml
curl http://localhost:8081/
```
- Acción:

    - Realizar una petición GET a la interfaz de OWASP ZAP.
- Resultado esperado:

    - La interfaz de ZAP debería cargar, indicando que está funcionando.

- <font color="red">**RECIBIDO:**</font>
```yaml
curl: (52) Empty reply from server
```

## **Test: Escaneo de vulnerabilidades**:<font color="red">**(POR DESARROLLAR)**</font>

- Acción: Ejecutar un escaneo desde OWASP ZAP contra el backend y el frontend.
- Resultado esperado: Un reporte con posibles vulnerabilidades de seguridad.

## Prueba ZAP dentro de contenedor: <font color="green">FUNCIONA</font>

In [ ]:
docker exec -it security curl http://localhost:8081/

```yaml
<head>
<title>ZAP API UI</title>
</head>
<body>
<h1>Welcome to the Zed Attack Proxy (ZAP)</h1><p>ZAP is an easy to use integrated penetration testing tool for finding vulnerabilities in web applications.</p><p></p><p>Please be aware that you should only attack applications that you have been specifically been given permission to test.</p><h2>Proxy Configuration</h2><p>To use ZAP effectively it is recommended that you configure your browser to proxy via ZAP.</p><p></p><p>The easiest way to do this is to launch your browser from ZAP via the "Quick Start / Manual Explore" panel - it will be configured to proxy via ZAP and ignore any certificate warnings.<br>Alternatively, you can configure your browser manually, or use the generated <a href="/OTHER/network/other/proxy.pac/?apinonce=713e2b884be29ec5">PAC file</a>.</p><h2>HTTPS Warnings Prevention</h2><p>To avoid HTTPS Warnings <a href="/OTHER/network/other/rootCaCert/?apinonce=16f1ceecc0346a7c">download</a> and <a href="https://www.zaproxy.org/docs/desktop/addons/network/options/servercertificates/#install" target="_blank">install CA root Certificate</a> in your Mobile device or computer.</p><h2>Links</h2><li><a href="/UI">Local API</a></li><li><a href="https://www.zaproxy.org/">ZAP Website</a></li><li><a href="https://groups.google.com/group/zaproxy-users">ZAP User Group</a></li><li><a href="https://groups.google.com/group/zaproxy-develop">ZAP Developer Group</a></li><li><a href="https://github.com/zaproxy/zaproxy/issues">Report an issue</a></li></body>
```

El resultado de `docker exec -it security curl http://localhost:8081/` muestra que:

ZAP está funcionando correctamente dentro del contenedor: El comando pudo conectarse al puerto 8081 y recibió una respuesta de la interfaz web de ZAP.

### Esto indica que el problema no está relacionado con el propio servicio ZAP dentro del contenedor.